<a href="https://colab.research.google.com/github/Victor-ro-dev/corn-pest-classifier/blob/main/treinamento_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌽 Classificador de Pragas do Milho — MobileNetV2 Customizada

Pipeline de treinamento supervisionado multiclasse para identificação de pragas na cultura do milho.

### 🎯 Mapeamento Oficial de Classes (Cultura do Milho):
1. **`lagarta_cartucho`** (*Spodoptera frugiperda*): Praga nº 1 — GBIF (larvas) + KaraAgro AI Maize (recortes de larva, danos e fezes).
2. **`lagarta_espiga`** (*Helicoverpa zea*): Decisiva para milho verde — GBIF (estágio larval, incl. CC BY-NC).
3. **`vaquinha`** (*Diabrotica speciosa*): Besouro com dano foliar inconfundível — GBIF (adultos).
4. **`milho_sadio`**: Classe negativa / controle sadio — PlantVillage / GBIF (*Zea mays*).

---

### 1. Configurando o Repositório no Colab (caso execute do zero)
Clona o repositório, entra no diretório do projeto e adiciona a pasta `src` ao path do Python:

In [ ]:
import os
import sys

# Clona o projeto se ainda não estiver clonado
if not os.path.exists('src'):
    if not os.path.exists('corn-pest-classifier'):
        !git clone https://github.com/Victor-ro-dev/corn-pest-classifier.git
    if os.path.exists('corn-pest-classifier'):
        %cd corn-pest-classifier

# Garante que a pasta src esteja acessível para importação em qualquer célula
if os.path.exists('src') and 'src' not in sys.path:
    sys.path.insert(0, os.path.abspath('src'))

print('Diretório ativo:', os.getcwd())
print('Python Path inclui src:', os.path.abspath('src') in sys.path)

### 2. Verificando o Acelerador de Hardware (GPU)
Certifique-se de que o ambiente de execução está configurado com GPU em: `Ambiente de execução` > `Alterar tipo de ambiente de execução` > `T4 GPU`.

In [ ]:
!nvidia-smi

### 3. Instalação das Dependências do Projeto

In [ ]:
!pip install -q -r requirements.txt

### 4. Coleta de Imagens de Pragas via GBIF
Baixa fotos de campo filtradas por estágio biológico para as 3 classes de pragas (`lagarta_cartucho`, `lagarta_espiga`, `vaquinha`):

In [ ]:
# Baixa fotos de campo organizadas por subpastas em dataset/<classe>
!python 01_baixar_gbif.py --max-por-classe 500 --saida dataset --modo subpastas --incluir-nc

### 5. Coleta de Amostras de Milho Sadio (PlantVillage ou GBIF)
Baixa centenas de folhas de milho sadias para a classe de controle negativo `milho_sadio`:

In [ ]:
# Opção PlantVillage (folhas sadias de controle):
!git clone --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset.git /tmp/plantvillage
!cd /tmp/plantvillage && git sparse-checkout set raw/color/Corn_\(maize\)___healthy
!mkdir -p dataset/milho_sadio
!cp -r /tmp/plantvillage/raw/color/Corn_\(maize\)___healthy/* dataset/milho_sadio/
!echo "Fotos de milho_sadio disponíveis:" $(ls dataset/milho_sadio | wc -l)

### 6. Verificação das 4 Classes Oficiais no Dataset

In [ ]:
import os
import sys
from pathlib import Path

# Adiciona 'src' ao sys.path para garantir que o módulo seja importado
for p in ['src', os.path.abspath('src')]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

try:
    from corn_pest_classifier.config import CLASS_NAMES
except ImportError:
    CLASS_NAMES = ['lagarta_cartucho', 'lagarta_espiga', 'vaquinha', 'milho_sadio']

dataset_path = Path('dataset')
print('=== VERIFICAÇÃO DO DATASET (CULTURA: MILHO) ===')
print(f'Classes oficiais esperadas ({len(CLASS_NAMES)}): {CLASS_NAMES}')

if dataset_path.exists():
    for c in CLASS_NAMES:
        pasta_classe = dataset_path / c
        if pasta_classe.exists() and pasta_classe.is_dir():
            imgs = [f for f in pasta_classe.glob('*') if f.is_file() and not f.name.endswith('.txt') and not f.name.endswith('.csv')]
            print(f'  [OK] {c}: {len(imgs)} imagens')
        else:
            print(f'  [PENDENTE] {c}: pasta não encontrada em dataset/{c}')
else:
    print('Diretório dataset/ não encontrado.')

### 7. Visualização de Amostras do Dataset & Data Augmentation

In [ ]:
# Gera as grades visuais do lote e do aumento de dados
!python src/corn_pest_classifier/visualizator.py

# Exibe os gráficos gerados diretamente no Colab
import os
from IPython.display import Image, display

if os.path.exists('preview_batch.png'):
    print('Amostras do lote original de treinamento:')
    display(Image('preview_batch.png'))

if os.path.exists('preview_augmentation.png'):
    print('Efeito das transformações de Data Augmentation:')
    display(Image('preview_augmentation.png'))

### 8. Treinamento da MobileNetV2 Customizada (4 Classes) e Conversão (.tflite)
Sincroniza os arquivos com o GitHub e inicia o treinamento com o número correto de classes:

In [ ]:
# Garante os scripts atualizados e inicia o treino
!git pull origin main
!python src/corn_pest_classifier/main.py

### 9. Baixar o Modelo Convertido (.tflite) Gerado

In [ ]:
import os
from google.colab import files

modelo_tflite = 'milho_pragas_model.tflite'
if os.path.exists(modelo_tflite):
    print('Iniciando download do modelo .tflite...')
    files.download(modelo_tflite)
else:
    print('Arquivo .tflite ainda não foi gerado.')